In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import pandas as pd
import time
import os
import re

# -----------------------------
# CONFIG
# -----------------------------
HEADERS = {"User-Agent": "Mozilla/5.0"}
CSV_FILE = "Tigray_online_news_dataset.csv"

session = requests.Session()
session.headers.update(HEADERS)

# -----------------------------
# RESUME LOGIC
# -----------------------------
processed_urls = set()

if os.path.exists(CSV_FILE):
    try:
        old = pd.read_csv(CSV_FILE)
        processed_urls = set(old["url"].dropna())
        print(f"Resuming... {len(processed_urls)} existing articles")
    except:
        print("Could not load previous file")

# -----------------------------
# ARCHIVES
# -----------------------------
archive_pages = [
    "https://www.tigraionline.com/archives.html"
]

archive_links = []

print("Fetching archives...")

for page in archive_pages:
    try:
        r = session.get(page, timeout=20)
        soup = BeautifulSoup(r.text, "html.parser")

        for a in soup.find_all("a", href=True):
            href = a["href"]

            if any(x in href.lower() for x in [
                "jan","feb","mar","apr","may","jun","jul","aug",
                "sep","oct","nov","dec","20"
            ]):
                archive_links.append({
                    "archive": a.get_text(strip=True),
                    "url": urljoin(page, href)
                })

    except:
        continue

archive_df = pd.DataFrame(archive_links).drop_duplicates("url")
print("Archives found:", len(archive_df))

# -----------------------------
# COLLECT ARTICLE LINKS
# -----------------------------
article_list = []

print("Collecting article links...")

for _, arch in archive_df.iterrows():
    try:
        r = session.get(arch["url"], timeout=20)
        soup = BeautifulSoup(r.text, "html.parser")

        for a in soup.find_all("a", href=True):
            url = urljoin(arch["url"], a["href"])
            parsed = urlparse(url)

            # keep only tigrai online
            if "tigraionline.com" not in parsed.netloc:
                continue

            if "/articles/" not in parsed.path:
                continue

            if url in processed_urls:
                continue

            article_list.append({
                "archive": arch["archive"],
                "url": url
            })

    except:
        continue

df_links = pd.DataFrame(article_list).drop_duplicates("url")

total = len(df_links)
print("New articles:", total)

# -----------------------------
# HELPERS
# -----------------------------
months = ("January|February|March|April|May|June|July|August|"
          "September|October|November|December")

date_pattern = rf"({months})\s+\d{{1,2}},\s+\d{{4}}"

def clean(text):
    return re.sub(r"\s+", " ", text).strip()

# -----------------------------
# SCRAPING
# -----------------------------
rows = []
saved_count = 0

print("Scraping articles...")

for i, row in df_links.iterrows():
    url = row["url"]

    try:
        r = session.get(url, timeout=20)
        soup = BeautifulSoup(r.text, "html.parser")

        container = (
            soup.find("div", id="art-big")
            or soup.find("div", id="article")
            or soup.find("div", class_="entry-content")
            or soup.find("article")
            or soup.body
        )

        if not container:
            continue

        raw_text = clean(container.get_text(" ", strip=True))

        # ---------------- TITLE ----------------
        title = "N/A"
        if container.find("h1"):
            title = clean(container.find("h1").get_text())
        elif soup.title:
            title = clean(soup.title.get_text())

        # ---------------- DATE ----------------
        date_match = re.search(date_pattern, raw_text)
        date = date_match.group(0) if date_match else ""

        # ---------------- AUTHOR ----------------
        author = ""

        m = re.search(
            r"(?:By|Written by|Reported by)\s+([A-Z][A-Za-z\s\.\-]{2,80})",
            raw_text
        )

        if m:
            author = m.group(1).strip()

        bad = ["few elite", "it was", "the", "in the"]
        if any(b in author.lower() for b in bad):
            author = ""

        # ---------------- CONTENT ----------------
        paragraphs = []
        for p in container.find_all("p"):
            txt = clean(p.get_text())
            if len(txt) > 30:
                paragraphs.append(txt)

        content = "\n\n".join(paragraphs) if paragraphs else raw_text

        rows.append({
            "url": url,
            "archive": row["archive"],
            "title": title,
            "date": date,
            "reported_by": author,
            "content": content
        })

        saved_count += 1

        # Save every 20 articles
        if saved_count % 20 == 0:
            df = pd.DataFrame(rows)

            df.to_csv(
                CSV_FILE,
                mode="a",
                header=not os.path.exists(CSV_FILE),
                index=False
            )

            rows = []

            print(f"Saved {saved_count}/{total}")

        time.sleep(1)

    except Exception:
        continue

# -----------------------------
# FINAL SAVE
# -----------------------------
if rows:
    df = pd.DataFrame(rows)

    df.to_csv(
        CSV_FILE,
        mode="a",
        header=not os.path.exists(CSV_FILE),
        index=False
    )

print("DONE ✅ Dataset saved:", CSV_FILE)

Fetching archives...
Archives found: 88
New articles: 2404
Scraping articles...
Saved 20/2404
Saved 40/2404
Saved 60/2404
Saved 80/2404
Saved 100/2404
Saved 120/2404
Saved 140/2404
Saved 160/2404
Saved 180/2404
Saved 200/2404
Saved 220/2404
Saved 240/2404
Saved 260/2404
Saved 280/2404
Saved 300/2404
Saved 320/2404
Saved 340/2404
Saved 360/2404
Saved 380/2404
Saved 400/2404
Saved 420/2404
Saved 440/2404
Saved 460/2404
Saved 480/2404
Saved 500/2404
Saved 520/2404
Saved 540/2404
Saved 560/2404
Saved 580/2404
Saved 600/2404
Saved 620/2404
Saved 640/2404
Saved 660/2404
Saved 680/2404
Saved 700/2404
Saved 720/2404
Saved 740/2404
Saved 760/2404
Saved 780/2404
Saved 800/2404
Saved 820/2404
Saved 840/2404
Saved 860/2404
Saved 880/2404
Saved 900/2404
Saved 920/2404
Saved 940/2404
Saved 960/2404
Saved 980/2404
Saved 1000/2404
Saved 1020/2404
Saved 1040/2404
Saved 1060/2404
Saved 1080/2404
Saved 1100/2404
Saved 1120/2404
Saved 1140/2404
Saved 1160/2404
Saved 1180/2404
Saved 1200/2404
Saved 1220/24

In [ ]:
import pandas as pd

df_read = pd.read_csv("Tigray_online_news_dataset.csv")
df= pd.DataFrame(df_read)
from IPython.display import display

display(df.head(20))

,url,archive,title,date,reported_by,content
0,https://www.tigraionline.com/articles/how-did-...,Feb-March 2018,Knowing the process how Dr Abiy is becoming a ...,"April 1, 2018",NaN,Dr Abiy’s sending to the premiership is a resu...
1,https://www.tigraionline.com/articles/challeng...,Feb-March 2018,Possible Challenges for the Newly Elected Prim...,"April 3, 2018",Professor Asayehgn Desta Tigrai Online,"By Professor Asayehgn Desta Tigrai Online, Apr..."
2,https://www.tigraionline.com/articles/dr-abiy-...,Feb-March 2018,EPRDF Council elects Dr. Abiy Ahmed as its Cha...,"March 27, 2018",NaN,GERD in the eyes of the Sudanese\n\nEthiopia’s...
3,https://www.tigraionline.com/articles/who-is-n...,Feb-March 2018,Who will be the next Ethiopian Prime Minister?...,"March 24, 2018",NaN,Who do you think will be the next Ethiopian Pr...
4,https://www.tigraionline.com/articles/ethiopia...,Feb-March 2018,The World Bank today approved $600 million in ...,"March 22, 2018",NaN,Ethiopia's Minister of Finance and Economic Co...
5,https://www.tigraionline.com/articles/size-tyr...,Feb-March 2018,Tyranny of Size in a Fragile Democracy: The Et...,"March 19, 2018",Mogos Asghedom Adwa Tigrai Online,"By Mogos Asghedom Adwa Tigrai Online, March 19..."
6,https://www.tigraionline.com/articles/vindicat...,Feb-March 2018,The vindication of the Weyanes in Ethiopia,"March 17, 2018",NaN,"N.D. Tedla Tigrai Online, March 17, 2018\n\nIt..."
7,https://www.tigraionline.com/articles/lufthans...,Feb-March 2018,Lufthansa to fly non-stop five times a week fr...,"March 21, 2018",NaN,"Lufthansa Press Release Addis Ababa, Tigrai On..."
8,https://www.tigraionline.com/articles/contest-...,Feb-March 2018,The Contest for The New Prime Minister And the...,"March 20, 2018",Orion Demame Tigrai Online,"By Orion Demame Tigrai Online, March 20, 2018\..."
9,https://www.tigraionline.com/articles/contest-...,Feb-March 2018,The Contest for The New Prime Minister And the...,"March 20, 2018",Orion Demame Tigrai Online,"By Orion Demame Tigrai Online, March 20, 2018\..."
